# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant pandas --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**Note:** All entities are referenced by their `@id` per FAIR^2 best practices.

In [ ]:
# List all record sets and their @id
record_sets = dataset.record_sets

print('Available Record Sets:')
for rs in record_sets:
    print(f"  - @id: {rs.id}, name: {getattr(rs, 'name', None)}")

# For each record set, list available fields/columns by their @id
for rs in record_sets:
    print(f"\nRecord Set @id: {rs.id}")
    print("Fields:")
    for field in getattr(rs, 'fields', []):
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview in the previous section.

In [ ]:
# ---[ You may need to customize record_set_ids based on the output above ]---
# Select all record sets
record_set_ids = [rs.id for rs in record_sets] # e.g., ['https://api.app.sen.science/frontiers/7154140/abcdef...', ...]
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading records for: {record_set_id}')
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- DataFrame shape: {df.shape}")
    print(f"- Columns (@id): {df.columns.tolist() if not df.empty else df.columns}")
    print('-'*32)

# For demonstration, select the first record set with data
main_record_set_id = None
for rid in record_set_ids:
    if not dataframes[rid].empty:
        main_record_set_id = rid
        break

print(f"Main record set for further analysis: {main_record_set_id}")
if main_record_set_id:
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on field values, normalizing numeric columns, and optionally grouping/categorizing data.

**Tip:** Use the field `@id` as column names in all DataFrame operations.

In [ ]:
import numpy as np

df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()
if not df.empty:
    # Find the first numerical field
    numeric_candidate_fields = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_candidate_fields:
        numeric_field = numeric_candidate_fields[0]
    else:
        # Try to convert suitable fields to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
                numeric_field = col
                break
            except:
                continue
        else:
            numeric_field = None
else:
    numeric_field = None

if numeric_field:
    print(f"Numeric field selected (by @id): {numeric_field}")
    threshold = df[numeric_field].quantile(0.5)  # median threshold as example
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Loaded {len(filtered_df)} records with {numeric_field} > {threshold}")
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    non_numeric_fields = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
    if non_numeric_fields:
        group_field = non_numeric_fields[0]
        print(f"\nGrouping by field (by @id): {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric fields found in the data for analysis.")

## 5. Visualization
Visualize field distributions or relationships. Here we plot the numeric field and grouped means, if available.

In [ ]:
import matplotlib.pyplot as plt

if not df.empty and numeric_field:
    # Histogram
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Grouped plot if group_field exists
    if 'group_field' in locals():
        grouped_df.plot(kind='bar', x=group_field, y=numeric_field, figsize=(10,4), legend=False, title=f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()
else:
    print("No numeric data available for plotting.")

## 6. Conclusion
- Successfully loaded the dataset metadata and records using `mlcroissant`.
- Explored the FAIR² dataset structure programmatically by referencing all fields by their `@id`.
- Performed basic EDA and visualized the main numerical field.

**Next steps:** You can repeat the analysis for other record sets and fields, or extend this notebook with modeling or statistical evaluation relevant to your research.